In [1]:
import numpy as np
import pandas as pd
import os
import xgboost as xgb

os.chdir(r"c:\Users\Huy\Documents\UIT\Sem_4\DS102")

In [2]:
X = pd.read_csv('data/split/feature.csv')
y = pd.read_csv('data/split/target.csv')
y = y[['SID', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'LAT', 'LON', 'TARGET_LAT_24h', 'TARGET_LON_24h']]

df = pd.merge(X, y, on=['SID', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'LAT', 'LON'], how='left')

In [3]:
# ==========================================
# PHẦN 1: HÀM ĐÁNH GIÁ SAI SỐ HAVERSINE
# ==========================================
def haversine_distance(lat_true, lon_true, lat_pred, lon_pred):
    """Tính khoảng cách thực tế (km) trên mặt cầu Trái Đất"""
    lat_true, lon_true, lat_pred, lon_pred = map(np.radians, [lat_true, lon_true, lat_pred, lon_pred])
    dlat = lat_pred - lat_true
    dlon = lon_pred - lon_true
    a = np.sin(dlat/2.0)**2 + np.cos(lat_true) * np.cos(lat_pred) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371 * c  # Bán kính Trái Đất R = 6371 km

In [4]:
# ==========================================
# PHẦN 2: CHUẨN BỊ VÀ CHIA TẬP DỮ LIỆU (TIME-BASED SPLIT)
# ==========================================
# Xóa các dòng có Target bị rỗng (NaN) khi bão kết thúc
df_clean = df.dropna(subset=['TARGET_LAT_24h', 'TARGET_LON_24h']).copy() 

# Chia tập dữ liệu dựa trên mốc thời gian (Năm) để chống Data Leakage tuyệt đối
train_df = df_clean[(df_clean['YEAR'] >= 2000) & (df_clean['YEAR'] <= 2020)]  # 21 years
val_df = df_clean[(df_clean['YEAR'] >= 2021) & (df_clean['YEAR'] <= 2022)]    # 2 years
test_df = df_clean[(df_clean['YEAR'] >= 2023) & (df_clean['YEAR'] <= 2024)]   # 2 years

# Định nghĩa các cột không phải là Feature (như mã bão, thời gian, tọa độ hiện tại và target)
drop_cols = ['SID', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'TARGET_LAT_24h', 'TARGET_LON_24h']

# Tách biến độc lập (X) và biến phụ thuộc (y)
X_train = train_df.drop(columns=drop_cols)
y_train_lat = train_df['TARGET_LAT_24h']
y_train_lon = train_df['TARGET_LON_24h']

X_val = val_df.drop(columns=drop_cols)
y_val_lat = val_df['TARGET_LAT_24h']
y_val_lon = val_df['TARGET_LON_24h']

X_test = test_df.drop(columns=drop_cols)
y_test_lat = test_df['TARGET_LAT_24h']
y_test_lon = test_df['TARGET_LON_24h']

# Giữ lại tọa độ hiện tại (T0) của tập Test để tính ngược ra vị trí tương lai trên bản đồ
current_lat_test = test_df['LAT'].values
current_lon_test = test_df['LON'].values

In [5]:
# ==========================================
# PHẦN 3: KHỞI TẠO VÀ HUẤN LUYỆN MÔ HÌNH GBDT
# ==========================================
# Cấu hình siêu tham số chuẩn theo nghiên cứu của Jinkai Tan để tránh Over-fitting
xgb_params = {
    'max_depth': 3,
    'learning_rate': 0.01,
    'n_estimators': 5000,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'objective': 'reg:squarederror',
    'tree_method': 'hist', # Tăng tốc bằng histogram
    'random_state': 42,
    'n_jobs': -1           # Dùng toàn bộ nhân CPU
}

print("Đang huấn luyện mô hình GBDT cho Vĩ độ (Latitude)...")
model_lat = xgb.XGBRegressor(**xgb_params, early_stopping_rounds=50)
model_lat.fit(
    X_train, y_train_lat, 
    eval_set=[(X_val, y_val_lat)], verbose=100
)

print("Đang huấn luyện mô hình GBDT cho Kinh độ (Longitude)...")
model_lon = xgb.XGBRegressor(**xgb_params, early_stopping_rounds=50)
model_lon.fit(
    X_train, y_train_lon,
    eval_set=[(X_val, y_val_lon)], verbose=100
)

Đang huấn luyện mô hình GBDT cho Vĩ độ (Latitude)...
[0]	validation_0-rmse:1.89433
[100]	validation_0-rmse:1.31477
[200]	validation_0-rmse:1.19522
[300]	validation_0-rmse:1.15417
[400]	validation_0-rmse:1.13694
[500]	validation_0-rmse:1.13015
[600]	validation_0-rmse:1.12590
[700]	validation_0-rmse:1.12140
[800]	validation_0-rmse:1.11922
[900]	validation_0-rmse:1.11641
[1000]	validation_0-rmse:1.11445
[1100]	validation_0-rmse:1.11218
[1200]	validation_0-rmse:1.11037
[1300]	validation_0-rmse:1.10943
[1400]	validation_0-rmse:1.10731
[1500]	validation_0-rmse:1.10504
[1600]	validation_0-rmse:1.10455
[1656]	validation_0-rmse:1.10432
Đang huấn luyện mô hình GBDT cho Kinh độ (Longitude)...
[0]	validation_0-rmse:3.42217
[100]	validation_0-rmse:1.98323
[200]	validation_0-rmse:1.61306
[300]	validation_0-rmse:1.50674
[400]	validation_0-rmse:1.47600
[500]	validation_0-rmse:1.46533
[600]	validation_0-rmse:1.45776
[700]	validation_0-rmse:1.45357
[800]	validation_0-rmse:1.45163
[900]	validation_0-rmse

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.7
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [6]:
# ==========================================
# PHẦN 4: DỰ BÁO VÀ ĐÁNH GIÁ TRÊN TẬP TEST
# ==========================================
# 1. Dự báo mức độ dịch chuyển (Delta Lat, Delta Lon)
pred_delta_lat = model_lat.predict(X_test)
pred_delta_lon = model_lon.predict(X_test)

# 2. Cộng mức độ dịch chuyển vào tọa độ T0 để ra tọa độ dự báo thực tế tại T+24h
pred_lat_24h = current_lat_test + pred_delta_lat
pred_lon_24h = current_lon_test + pred_delta_lon

true_lat_24h = current_lat_test + y_test_lat.values
true_lon_24h = current_lon_test + y_test_lon.values

# 3. Tính toán sai số khoảng cách bằng Haversine
test_errors_km = haversine_distance(true_lat_24h, true_lon_24h, pred_lat_24h, pred_lon_24h)
mean_error = np.mean(test_errors_km)

print(f"\n=> SAI SỐ KHOẢNG CÁCH DỰ BÁO 24H TRUNG BÌNH (Mean Forecast Error): {mean_error:.2f} km")


=> SAI SỐ KHOẢNG CÁCH DỰ BÁO 24H TRUNG BÌNH (Mean Forecast Error): 144.49 km


In [7]:
import numpy as np
import pandas as pd

def evaluate_advanced_metrics(y_true, y_pred, v_past):
    """
    y_true: Mảng chứa [delta_lat_obs, delta_lon_obs]
    y_pred: Mảng chứa [delta_lat_pred, delta_lon_pred]
    v_past: Vận tốc bão trong 24h qua (V_Actual_24h từ dữ liệu gốc)
    """
    
    # Chuyển đổi sang numpy array để tính toán vector nhanh
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    v_past = np.array(v_past)
    
    # Tách biệt Delta Lat (Y) và Delta Lon (X)
    dY_obs, dX_obs = y_true[:, 0], y_true[:, 1]
    dY_pred, dX_pred = y_pred[:, 0], y_pred[:, 1]
    
    # ---------------------------------------------------------
    # 2. DIRECTIONAL STABILITY (Độ ổn định hướng)
    # ---------------------------------------------------------
    # Điều kiện đúng: Cùng dấu (Tích > 0) cho cả vĩ độ và kinh độ
    correct_direction = (dX_obs * dX_pred > 0) & (dY_obs * dY_pred > 0)
    directional_stability = np.mean(correct_direction) * 100
    
    # ---------------------------------------------------------
    # 3. MOVING VELOCITY SENSITIVITY (Độ nhạy vận tốc)
    # ---------------------------------------------------------
    # Tính vận tốc dự báo và thực tế trong 24h tới (V_next)
    # Công thức: sqrt(delta_lat^2 + delta_lon^2) / thời gian
    # Ở đây dùng đơn vị tương đối (quãng đường dịch chuyển) để so sánh tỷ lệ
    v_next_obs = np.sqrt(dX_obs**2 + dY_obs**2)
    v_next_pred = np.sqrt(dX_pred**2 + dY_pred**2)
    
    # Tránh lỗi chia cho 0 nếu bão đứng yên
    v_past = np.where(v_past == 0, 0.001, v_past)
    
    # Xác định các ca biến động mạnh (Ground Truth)
    accel_cases = (v_next_obs / v_past > 2.0)
    decel_cases = (v_next_obs / v_past < 0.5)
    
    # Kiểm tra mô hình có bắt trúng không
    accel_hit = accel_cases & (v_next_pred / v_past > 2.0)
    decel_hit = decel_cases & (v_next_pred / v_past < 0.5)
    
    # Tính tỷ lệ nhạy (Sensitivity)
    total_anomalies = np.sum(accel_cases) + np.sum(decel_cases)
    total_hits = np.sum(accel_hit) + np.sum(decel_hit)
    
    # Tránh chia cho 0 nếu tập test không có ca đột biến nào
    velocity_sensitivity = (total_hits / total_anomalies * 100) if total_anomalies > 0 else 0
    
    # In kết quả
    print("--- KẾT QUẢ ĐÁNH GIÁ CHUYÊN SÂU ---")
    print(f"1. Directional Stability: {directional_stability:.2f}%")
    print(f"   (Kỳ vọng: > 87%)")
    print(f"2. Moving Velocity Sensitivity: {velocity_sensitivity:.2f}%")
    print(f"   (Kỳ vọng: ~ 40-60% - Đây là điểm yếu của GBDT)")
    print(f"   - Số ca đột biến thực tế: {total_anomalies}")
    print(f"   - Số ca mô hình bắt trúng: {total_hits}")

    return directional_stability, velocity_sensitivity


v_past = test_df['V_Actual_24h'].values
actual = np.column_stack((y_test_lat, y_test_lon))
y_test_actual = pd.DataFrame(actual)

predict = np.column_stack((pred_delta_lat, pred_delta_lon))
y_test_predicted = pd.DataFrame(predict)

evaluate_advanced_metrics(y_test_actual, y_test_predicted, v_past)

--- KẾT QUẢ ĐÁNH GIÁ CHUYÊN SÂU ---
1. Directional Stability: 85.55%
   (Kỳ vọng: > 87%)
2. Moving Velocity Sensitivity: 16.67%
   (Kỳ vọng: ~ 40-60% - Đây là điểm yếu của GBDT)
   - Số ca đột biến thực tế: 84
   - Số ca mô hình bắt trúng: 14


(np.float64(85.54913294797689), np.float64(16.666666666666664))